# Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo


# Chargement des données


In [ ]:
wine_quality = fetch_ucirepo(id=186)

X = wine_quality.data.features
y = wine_quality.data.targets

df = pd.concat([X, y], axis=1)
df.head()


# Métadonnées et variables


In [ ]:
print(wine_quality.metadata)
print(wine_quality.variables)


# Structure du jeu de données


In [ ]:
print(df.shape)
df.info()


# Statistiques descriptives


In [ ]:
df.describe().round(2)


# Vérification des valeurs manquantes


In [ ]:
df.isna().sum()


# Détection des valeurs aberrantes par l'écart interquartile


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop("quality")
q1 = df[numeric_cols].quantile(0.25)
q3 = df[numeric_cols].quantile(0.75)
iqr = q3 - q1
outliers = ((df[numeric_cols] < (q1 - 1.5 * iqr)) | (df[numeric_cols] > (q3 + 1.5 * iqr))).sum()
outliers


# Proportion des valeurs aberrantes


In [ ]:
(outliers / len(df) * 100).round(2)


# Encodage de la variable catégorielle


In [ ]:
if "color" in wine_quality.data.original.columns:
    df["color"] = wine_quality.data.original["color"].map({"red": 1, "white": 0})
df["color"].value_counts()


# Création de la cible binaire


In [ ]:
df["good_wine"] = (df["quality"] >= 7).astype(int)
df[["quality", "good_wine"]].head()


# Répartition des notes de qualité d'origine


In [ ]:
df["quality"].value_counts().sort_index()


# Distribution de la cible binaire


In [ ]:
counts = df["good_wine"].value_counts()
proportions = (df["good_wine"].value_counts(normalize=True) * 100).round(2)
pd.DataFrame({"effectif": counts, "pourcentage": proportions})


# Visualisation du déséquilibre des classes


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df["quality"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#2b5c8f")
axes[0].set_title("Scores de qualité")
axes[0].set_xlabel("Qualité")
axes[0].set_ylabel("Effectif")

df["good_wine"].value_counts().plot(kind="bar", ax=axes[1], color=["#b83b5e", "#6a2c70"])
axes[1].set_title("Cible binaire")
axes[1].set_xlabel("Bon vin (>= 7)")
axes[1].set_ylabel("Effectif")
axes[1].set_xticklabels(["0 (Ordinaire)", "1 (Bon)"], rotation=0)

plt.tight_layout()
plt.show()


# Matrice de corrélation


In [ ]:
corr_matrix = df.corr().round(2)
corr_matrix


# Corrélations avec la cible binaire


In [ ]:
target_corr = corr_matrix["good_wine"].sort_values(ascending=False)
target_corr


# Visualisation de la matrice de corrélation


In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        plt.text(j, i, corr_matrix.iloc[i, j], ha="center", va="center", color="black", fontsize=7)
plt.title("Matrice de corrélation des variables")
plt.tight_layout()
plt.show()


# Distributions des variables physico-chimiques


In [ ]:
features = [col for col in df.columns if col not in ["quality", "good_wine"]]
df[features].hist(bins=20, figsize=(14, 10), color="#2b5c8f", edgecolor="black", layout=(4, 3))
plt.tight_layout()
plt.show()


# Boxplots des variables selon la qualité du vin


In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(14, 12))
axes = axes.flatten()

for i, col in enumerate(features):
    data_to_plot = [df[df["good_wine"] == 0][col].dropna(), df[df["good_wine"] == 1][col].dropna()]
    axes[i].boxplot(data_to_plot, patch_artist=True, boxprops=dict(facecolor="#95a5a6"), medianprops=dict(color="black"))
    axes[i].set_title(col)
    axes[i].set_xticklabels(["Ordinaire (0)", "Bon (1)"])

for j in range(len(features), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


# Définition des variables explicatives et de la cible


In [ ]:
X_data = df.drop(columns=["quality", "good_wine"])
y_data = df["good_wine"]


# Séparation des données en ensembles d'entraînement et de test


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
)
print(X_train.shape, X_test.shape)


# Standardisation des variables numériques


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Sélection univariée avec SelectKBest et le test ANOVA


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k="all")
selector.fit(X_train_scaled, y_train)

scores_df = pd.DataFrame({
    "variable": X_data.columns,
    "score_f": selector.scores_.round(2),
    "p_value": selector.pvalues_
}).sort_values(by="score_f", ascending=False)
scores_df


# Classement graphique des variables selon le score F


In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(scores_df["variable"][::-1], scores_df["score_f"][::-1], color="#2b5c8f")
plt.xlabel("Score F (ANOVA)")
plt.title("Importance des variables selon le test F d'ANOVA")
plt.tight_layout()
plt.show()


# Régression Logistique : entraînement et prédiction


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

lr_model = LogisticRegression(solver="liblinear", random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]


# Métriques de performance de la Régression Logistique


In [ ]:
lr_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_lr),
    "Précision": precision_score(y_test, y_pred_lr),
    "Rappel": recall_score(y_test, y_pred_lr),
    "F1-score": f1_score(y_test, y_pred_lr),
    "ROC-AUC": roc_auc_score(y_test, y_prob_lr)
}
pd.Series(lr_metrics).round(4)


# Matrice de confusion de la Régression Logistique


In [ ]:
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr, target_names=["Ordinaire (0)", "Bon (1)"]))


# k-NN : entraînement et prédiction


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

y_pred_knn = knn_model.predict(X_test_scaled)
y_prob_knn = knn_model.predict_proba(X_test_scaled)[:, 1]


# Métriques de performance de k-NN


In [ ]:
knn_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_knn),
    "Précision": precision_score(y_test, y_pred_knn),
    "Rappel": recall_score(y_test, y_pred_knn),
    "F1-score": f1_score(y_test, y_pred_knn),
    "ROC-AUC": roc_auc_score(y_test, y_prob_knn)
}
pd.Series(knn_metrics).round(4)


# Matrice de confusion de k-NN


In [ ]:
print(confusion_matrix(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn, target_names=["Ordinaire (0)", "Bon (1)"]))
